# **Riconoscere automaticamente il livello CEFR di un testo: un confronto tra metodi a regole, apprendimento automatico e LLM**
Il progetto ha come obiettivo il classificare automaticamente un testo in base al livello linguistico, facendo riferimento al Quadro Comune Europeo di Riferimento per le lingue (*QCER*), in inglese *CEFR* (Common European Framework of Reference for languages). Verranno esplorati vari metodi, raggruppati in 3 approcci differenti (*Rule-Based, Machine Learning* e *LLM*), e ne verrà valutata la performance su un dataset di testing.




## **Introduzione, gestione dei dataset e preprocessing**
**NoteBook 1/3.**

#**1. Introduzione**

##**1.a. Il sistema *CEFR*** :
Il *Common European Framework of Reference for Languages* ("QCER", in italiano), è un

>sistema descrittivo impiegato per valutare le conoscenze e competenze conseguite da chi studia una lingua straniera europea (e potenzialmente qualunque altra lingua nel mondo), nonché allo scopo di indicare il livello di un insegnamento linguistico negli ambiti più disparati. [[1]](https://it.wikipedia.org/wiki/Quadro_comune_europeo_di_riferimento_per_la_conoscenza_delle_lingue#Nascita_del_QCER:~:text=%C3%A8%20un%20sistema%20descrittivo%20impiegato%20per%20valutare%20le%20conoscenze%20e%20competenze%20conseguite%20da%20chi%20studia%20una%20lingua%20straniera%20europea%20(e%20potenzialmente%20qualunque%20altra%20lingua%20nel%20mondo)%2C%20nonch%C3%A9%20allo%20scopo%20di%20indicare%20il%20livello%20di%20un%20insegnamento%20linguistico%20negli%20ambiti%20pi%C3%B9%20disparati.)

Il sistema è diviso in 6 livelli principali, **A1,A2,B1,B2,C1** e **C2**, i quali indicano una serie di competenze crescenti, divisi a loro volta per i "macro-livelli" **A**, **B** e **C**. Le abilità linguistiche sono 4: comprensione scritta e orale e produzione scritta e orale. Le varie abilità sono così definite:
  User|Level|Description
  ---|---|---
  Basic|A1|Can understand and use familiar everyday expressions and very basic phrases aimed at the satisfaction<br>of needs of a concrete type. Can introduce him/herself and others and can ask and answer questions about<br>personal details such as where he/she lives, people he/she knows and things he/she has. Can interact in a<br>simple way provided the other person talks slowly and clearly and is prepared to help.
  Basic|A2|Can understand sentences and frequently used expressions related to areas of most immediate relevance<br>(e.g. very basic personal and family information, shopping, local geography, employment). Can communicate<br>in simple and routine tasks requiring a simple and direct exchange of information on familiar and routine matters.<br>Can describe in simple terms aspects of his/her background, immediate environment and matters in areas<br>of immediate need.
  Intermediate|B1|Can understand the main points of clear standard input on familiar matters regularly encountered in work, school,<br>leisure, etc. Can deal with most situations likely to arise whilst travelling in an area where the language is spoken.<br>Can produce simple connected text on topics which are familiar or of personal interest. Can describe experiences<br>and events, dreams, hopes & ambitions and briefly give reasons and explanations for opinions and plans.
  Intermediate|B2|	Can understand the main ideas of complex text on both concrete and abstract topics, including technical<br>discussions in his/her field of specialisation. Can interact with a degree of fluency and spontaneity that<br>makes regular interaction with native speakers quite possible without strain for either party. Can produce<br>clear, detailed text on a wide range of subjects and explain a viewpoint on a topical issue giving the<br>advantages and disadvantages of various options.
  Proficient|C1|Can understand a wide range of demanding, longer texts, and recognise implicit meaning. Can express<br>him/herself fluently and spontaneously without much obvious searching for expressions. Can use language<br> flexibly and effectively for social, academic and professional purposes. Can produce clear, well-structured,<br>detailed text on complex subjects, showing controlled use of organisational patterns, connectors and <br>cohesive devices.
  Proficient|C2|Can understand with ease virtually everything heard or read. Can summarise information from different spoken<br>and written sources, reconstructing arguments and accounts in a coherent presentation. Can express him/herself<br>spontaneously, very fluently and precisely, differentiating finer shades of meaning even in more complex situations.
  
   [[2]](https://www.coe.int/en/web/common-european-framework-reference-languages/table-1-cefr-3.3-common-reference-levels-global-scale#:~:text=PROFICIENT%20USER,prepared%20to%20help.)



##**1.b. *Text Classification***
La Text Classification è un task che consiste nell'assegnare automaticamente un'etichetta ("label") ad un testo a partire da un determinato insieme di etichette. [[3]](https://www.sciencedirect.com/topics/mathematics/text-classification).

Esistono 3 tipi principali di *TC*:

1.	*Binary*: 2 etichette “opposte” disponibili: testo spam/not-spam, oppure testo in inglese/non-inglese;

2.	*Multiclass*: 3 o più etichette disponibili e dobbiamo scegliere quale sia la più adatta ;
3.	*Multi-label*: 3 o più etichette ma possiamo assegnarne più di una ad un’unità di testo: ad esempio, dobbiamo assegnare ad un autore (come Dante) i generi testuali prodotti: poema, raccolta, romanzo, trattato e altri.

Nel nostro caso ci concentreremo nella classificazione *multiclass*, con etichette da A1 a C1 (C2 è stata esclusa per mancanza di materiale testuale).

##**1.c. *Dataset***

I vari dataset sui quali verrà svolto il progetto sono divisi in due gruppi:


1.   [EFCamDat](https://ef-lab.mmll.cam.ac.uk/EFCAMDAT.html) : un corpus composto da due file Excel contenenti saggi brevi ("Essay") etichettati con livelli linguistici CEFR, dall'A2 al C1, svolti da studenti durante gli esami di certificazione presso le sedi online di EF Schools. I due testi sono stati presi dal "Cleaned Subcorpus".   

2.   [UniversalCEFR](https://huggingface.co/UniversalCEFR) : una serie di dataset di testi etichettati, che verranno utilizzati per capire come generalizzano i vari modelli;

3. Alcuni file contenenti delle parole etichettate, saranno utilizzati nel Notebook di training (2) per il modello "Raw Counts".



### Importazione delle librerie

In [ ]:
import pandas as pd
import json
import random
import os
from pathlib import Path
import spacy
import re
import gdown
import zipfile
nlp = spacy.load("en_core_web_sm")

### Download dei vari dataset

In [ ]:
url = 'https://drive.google.com/uc?id=1nCKM24zASgU2poqp1L7LpRiiSQCq5prI'
gdown.download(url, 'material_notebook_1.zip',quiet=True, use_cookies=False)

'material_notebook_1.zip'

In [ ]:
zip_file_path = r'/content/material_notebook_1.zip'
extract_dir = r'/content/'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

### Creazione directory di output
Dato che i vari modelli funzionano diversamente in base al tipo di preprocessing al quale è sottoposto il dataset di training, vengono create delle cartelle che conterranno i vari dataset e i vari split di training e testing.

In [ ]:
bases = [Path("preprocessed/main_dataset"), Path("preprocessed/generalized_dataset")]
for base in bases:
  subdirs = [
      base,
      base / "0.raw",
      base / "1.lowercased",
      base / "2.lowercased_lemmatized",
      base / "3.lowercased_nostopwords",
      base / "4.lowercased_nostopwords_lemmatized",
      base / "5.lowercased_nopunctuation",
      base / "6.lowercased_nopunctuation_lemmatized",
      base / "7.lowercased_nopunctuation_nostopwords",
      base / "8.lowercased_nopunctuation_nostopwords_lemmatized",
  ]

  for d in subdirs:
      d.mkdir(parents=True, exist_ok=True)

I percorsi di entrambi i gruppi di dataset vengono raccolti in una lista

In [ ]:
main_texts_datasets = [
    r"/content/material_notebook_1/datasets_efcamdat/Final database (main prompts).xlsx",
    r"/content/material_notebook_1/datasets_efcamdat/Final database (alternative prompts).xlsx"
]

generalization_texts_datasets = []
for filename in [file for file in os.listdir(r'/content/material_notebook_1/datasets_for_generalization') if file.endswith('.json')]:
  generalization_texts_datasets.append(os.path.join('/content/material_notebook_1/datasets_for_generalization',filename))


###**Estrazione dei testi *EFCamDat***

Il file Excel è composto da varie colonne, e tramite *pandas* è possibile estrarne due specifiche:
1. *cefr* ;
2. *text_corrected* ;

Servono circa 4 minuti:

In [ ]:
df1 = pd.read_excel(
    main_texts_datasets[0],
    usecols=['cefr', 'text_corrected']
)
df2 = pd.read_excel(
    main_texts_datasets[1],
    usecols=['cefr', 'text_corrected']
)

I due dataset vengono uniti con *pd.concat()* e vengono rimossi i duplicati.

In [ ]:
final_df = pd.concat([df1, df2], ignore_index=True)
final_df = final_df.drop_duplicates(subset='text_corrected')

Tramite una **dictionary comprehension** viene creato un dizionario con i livelli come chiavi (secondo la struttura "*__label__LIVELLO* "), e una lista di testi come valori.

In [ ]:
data = {
    f'__label__{lvl}': final_df.loc[final_df['cefr'] == lvl, 'text_corrected'].tolist() for lvl in ['A1', 'A2', 'B1', 'B2', 'C1']
}

Viene stampato il primo testo per ogni livello

In [ ]:
print(data['__label__A1'][0])
print(data['__label__A2'][0])
print(data['__label__B1'][0])
print(data['__label__B2'][0])
print(data['__label__C1'][0])

Hi, My name's Leon. I don't speak english, but I am in learning stage, so I do not have much to say in the newsroom. Good, study the 3rd grade of high school and I intend to exchange. Good Night, bye.
I work in a factory, and I am a sales assistant. sometimes I think my job is really interesting, I always talk on phone and write emails, sometimes go to a business trip. I always work long time and have little breaks. When finished work to home I feel tired.
This game is called Frisbee howling. The rules are easy: we have the bowling alley, an area of about 8 meters x by 3 meters; the bowling pins, composed of ten plastic bottles lined up in rows of 4, then 3, then 1. each player must knock down all the bottles with two shots using a frisbee. One point for each bottle knocked down. There are ten turns and all the points are added together. The player with most points is the winner! Remember, isn't easy play this game with much wind!
Hi, Anna I hear that you are finding a job, and here is

Per prevenire uno sbilanciamento nelle varie classi (quindi per evitare che alcune classi abbiano molti più testi di altre), viene creato un dizionario *amounts* e tramite un ciclo *for* vengono inserite le varie etichette, corrispondenti al numero di testi per quella etichetta.
A *smallest_dataset* viene assegnata l'etichetta con il dataset più piccolo (in questo caso, *__label__C1*), e viene assegnato alla variabile *index* il valore rispondente alla chiave di *smallest_dataset*.

In [ ]:
amounts = {}
for key in data.keys():
    amounts[key] = len(data[key])

smallest_dataset = min(amounts, key=amounts.get)
print(smallest_dataset)

index = amounts[smallest_dataset]
print(index)

__label__C1
9990


Vengono poi inseriti i vari testi in *data*, quindi per ogni chiave, dopo essere stati mescolati con *random.shuffle()*. Il numero di testi è 9990, l'indice estratto nella cella precedente. Nella lista vengono inseriti i testi, come stringhe, secondo una struttura:
        
    "__label__LIVELLO this is a text..."


In [ ]:
labelled_texts = []
for key in data.keys():
    random.shuffle(data[key])
    for text in data[key][:index]:
        labelled_texts.append(f'{key} {text}\n')

I testi vengono mescolati e scritti su file:

In [ ]:
random.shuffle(labelled_texts)

with open(r'/content/preprocessed/main_dataset/0.raw/main_dataset.txt', encoding='utf-8', mode='w') as file:
    file.writelines(labelled_texts)

###Testi per generalizzare
Per testare se i modelli generalizzano bene, viene creato un file con i vari dataset di UniversalCEFR, seguendo un approccio simile a quello precedente.

Servono circa 2 minuti:

In [ ]:
output = []
n = 1
for sample in os.listdir(r'/content/material_notebook_1/datasets_for_generalization'):
    with open(os.path.join(r'/content/material_notebook_1/datasets_for_generalization',sample),'r', encoding='utf-8') as testing_file:
        try:
            content = json.load(testing_file)
        except json.decoder.JSONDecodeError:
            continue
        for item in content:
            if item['cefr_level'] != 'C2':
                preprocessed_text = item['text'].replace('\n',' ')
                output.append(f'__label__{item["cefr_level"][:2]} {preprocessed_text}\n')
                n = n+1
            random.shuffle(output)
            with open(os.path.join(r'/content/preprocessed/generalized_dataset/0.raw','generalized_dataset.test'),'w',encoding='utf-8') as testing_file_out:
                testing_file_out.writelines(output)

###**Preprocessing**
Dato che i vari "modelli" funzionano meglio o peggio in base al preprocessing, viene creata una funzione che, dopo aver attuato una serie di step di preprocessing di default, come:
* testo in minuscolo;
* rimozione indirizzi e-mail;
* rimozione numeri attaccati alle parole (alcuni testi presentano infatti stringhe del tipo "testo123");
* numeri vari;

prende come argomenti dei valori booleani e permette varie combinazioni di preprocessing, come:
* rimozione della punteggiatura;
* lemmatizzazione;
* rimozione delle stopwords.

In [ ]:
def advanced_preprocessing(text, do_remove_punctuation=True, do_lemmatize=True, do_stopwords_removal=True):

    text = text.lower()
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'([a-zA-Z])(\d+)', r'\1 ', text)
    text = re.sub(r'\d+', '', text)

    if do_remove_punctuation:
      text = re.sub(r'[^\w\s]', '', text)
      text = re.sub(r',', ' ', text)

    doc = nlp(text)
    tokens = []
    for token in doc:
        text = token.lemma_ if do_lemmatize else token.text
        text = text.strip()

        if text.isalpha() and len(text) > 1:
            if not do_stopwords_removal or not token.is_stop:
                tokens.append(text)
    return re.sub(r'\b(?![aAi])[A-Za-z]\b', '', ' '.join(tokens))

Viene creata una lista di dizionari contenente una serie di percorsi di output e la relativa combinazione di valori booleani. Ad esempio, il percorso '/content/preprocessed/main_dataset/2.lowercased_lemmatized/main_dataset.txt' conterrà una combinazione di booleani *False, True, False*.

####**Preprocessing per i testi per il training/testing**

In [ ]:
output_paths_and_combinations = [
    {
        'path':r'/content/preprocessed/main_dataset/0.raw/main_dataset.txt',
        'combination':[False,False,False]
    },
    {
        'path':r'/content/preprocessed/main_dataset/1.lowercased/main_dataset.txt',
        'combination':[False,False,False]
    },
    {
        'path':r'/content/preprocessed/main_dataset/2.lowercased_lemmatized/main_dataset.txt',
        'combination':[False,True,False]
    },
    {
        'path':r'/content/preprocessed/main_dataset/3.lowercased_nostopwords/main_dataset.txt',
        'combination':[False,False,True]
    },
    {
        'path':r'/content/preprocessed/main_dataset/4.lowercased_nostopwords_lemmatized/main_dataset.txt',
        'combination':[False,True,True]
    },
    {
        'path':r'/content/preprocessed/main_dataset/5.lowercased_nopunctuation/main_dataset.txt',
        'combination':[True,False,False]
    },
    {
        'path':r'/content/preprocessed/main_dataset/6.lowercased_nopunctuation_lemmatized/main_dataset.txt',
        'combination':[True,True,False]
    },
    {
        'path':r'/content/preprocessed/main_dataset/7.lowercased_nopunctuation_nostopwords/main_dataset.txt',
        'combination':[True,False,True]
    },
    {
        'path':r'/content/preprocessed/main_dataset/8.lowercased_nopunctuation_nostopwords_lemmatized/main_dataset.txt',
        'combination':[True,True,True]
    }
]

Viene caricato il file con i testi non preprocessati

In [ ]:
with open(r'/content/preprocessed/main_dataset/0.raw/main_dataset.txt',mode='r',encoding='utf-8') as file_:
    content = file_.readlines()

Vengono creati tutti i file preprocessati. Dato che sono circa 50k testi per 8 combinazioni, la cella impiega diversi minuti per completare il preprocessing:

In [ ]:
for pair in output_paths_and_combinations:

  output = []

  partial_path = '/'.join(pair['path'].split('/')[:-1]) #estraiamo un percorso parziale (cioè il percorso senza il nome del file)
  path = pair['path'] #"estraiamo" il percorso completo (parziale + nome del file)
  do_remove_punctuation, do_lemmatize, do_stopwords_removal = pair['combination'] #settiamo i valori booleani per le varie combinazioni

  n = 1
  if 'raw' not in path: #evitiamo di preprocessare i testi "raw"
    for row in content:
        label, text = row.split(' ',maxsplit=1)
        tokens = advanced_preprocessing(text,do_remove_punctuation=do_remove_punctuation, do_lemmatize=do_lemmatize, do_stopwords_removal=do_stopwords_removal)
        if text:
          output.append(f'{label} {tokens}\n')
        else:
          continue
        if n % 50 == 0:
          print(f'Split: {partial_path}.    Text {n} out of {len(content)}') #tramite questo "print()" monitoriamo l'andamento del preprocessing
        n = n+1

  #Determiniamo una split 80/20
  training_split = output[:round(len(output)*0.8)]
  testing_split = output[round(len(output)*0.8):]

  #Per ogni percorso, scriviamo il file intero e i due file di split che ne derivano:
  #Full file
  with open(path,mode='w',encoding='utf-8') as file_:
    file_.writelines(output)

  #Training split
  with open(os.path.join(partial_path,'main_dataset.train'), 'w', encoding='utf-8') as training_file:
     training_file.writelines(training_split)

  #Testing split
  with open(os.path.join(partial_path,'main_dataset.test'), 'w', encoding='utf-8') as testing_file:
     testing_file.writelines(testing_split)


####Preprocessing per i testi per generalizzare
Anche qui vengono determinati i percorsi e le combinazioni di booleani

In [ ]:
generalized_output_paths_and_combinations = [
    {
        'path':r'/content/preprocessed/generalized_dataset1/.lowercased/generalized_dataset.test',
        'combination':[False,False,False]
    },
    {
        'path':r'/content/preprocessed/generalized_dataset/2.lowercased_lemmatized/generalized_dataset.test',
        'combination':[False,True,False]
    },
    {
        'path':r'/content/preprocessed/generalized_dataset/3.lowercased_nostopwords/generalized_dataset.test',
        'combination':[False,False,True]
    },
    {
        'path':r'/content/preprocessed/generalized_dataset/4.lowercased_nostopwords_lemmatized/generalized_dataset.test',
        'combination':[False,True,True]
    },
    {
        'path':r'/content/preprocessed/generalized_dataset/5.lowercased_nopunctuation/generalized_dataset.test',
        'combination':[True,False,False]
    },
    {
        'path':r'/content/preprocessed/generalized_dataset/6.lowercased_nopunctuation_lemmatized/generalized_dataset.test',
        'combination':[True,True,False]
    },
    {
        'path':r'/content/preprocessed/generalized_dataset/7.lowercased_nopunctuation_nostopwords/generalized_dataset.test',
        'combination':[True,False,True]
    },
    {
        'path':r'/content/preprocessed/generalized_dataset/8.lowercased_nopunctuation_nostopwords_lemmatized/generalized_dataset.test',
        'combination':[True,True,True]
    }
]

E nuovamente vengono preprocessati i testi, come nel dataset training/testing. Qui non viene creata nessuna split, ma un file unico.

In [ ]:
with open(r'/content/preprocessed/generalized_dataset/0.raw/generalized_dataset.test',mode='r',encoding='utf-8') as file_:
    content = file_.readlines()

for pair in generalized_output_paths_and_combinations:

  output = []

  partial_path = '/'.join(pair['path'].split('/')[:-1]) #estraiamo un percorso parziale (cioè il percorso senza il nome del file)
  path = pair['path'] #"estraiamo" il percorso completo (parziale + nome del file)
  do_remove_punctuation, do_lemmatize, do_stopwords_removal = pair['combination'] #settiamo i valori booleani per le varie combinazioni

  n = 1
  for row in content:
      label, text = row.split(' ',maxsplit=1)
      tokens = advanced_preprocessing(text,do_remove_punctuation=do_remove_punctuation, do_lemmatize=do_lemmatize, do_stopwords_removal=do_stopwords_removal)
      if text:
        output.append(f'{label} {tokens}\n')
        if n % 50 == 0:
          print(f'Type: {path.split("/")[-2]}    --> Text: {n}/{len(content)}')
        n = n+1
      else:
        continue
  #Testing split
  with open(os.path.join(partial_path,'generalized_dataset.test'), 'w', encoding='utf-8') as testing_file:
    testing_file.writelines(output)

Type: .lowercased    --> Text: 50/5665
Type: .lowercased    --> Text: 100/5665
Type: .lowercased    --> Text: 150/5665
Type: .lowercased    --> Text: 200/5665
Type: .lowercased    --> Text: 250/5665
Type: .lowercased    --> Text: 300/5665


KeyboardInterrupt: 

##Parole etichettate
Tramite vari file vengono caricate le varie parole in un unico dizionario, *output*. Alcune parole, avendo una doppia ortografia, erano salvate in un'unica riga, come nel caso di *"acknowledgment/acknowledgement": "B1"*: queste sono state divise in due parole diverse; quindi:
* "acknowledgment: "B1", "acknowledgement": "B1".

In [ ]:
output = {}

for file in [f_ for f_ in os.listdir(r'/content/unprocessed_datasets/words') if f_.endswith('.json')]:
    file_path = os.path.join(r'/content/unprocessed_datasets/words', file)

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

        for key in data.keys():
            if '/' in key:
                versions = key.split('/')
                for version in versions:
                    output[version] = data[key]
            else:
                output[key] = data[key]


with open(r'labelled_words.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=4)

#**Note**
Nota|Nome|URL
----|------|-------
[1]|CEFR|[URL](https://it.wikipedia.org/wiki/Quadro_comune_europeo_di_riferimento_per_la_conoscenza_delle_lingue#Nascita_del_QCER:~:text=%C3%A8%20un%20sistema%20descrittivo%20impiegato%20per%20valutare%20le%20conoscenze%20e%20competenze%20conseguite%20da%20chi%20studia%20una%20lingua%20straniera%20europea%20(e%20potenzialmente%20qualunque%20altra%20lingua%20nel%20mondo)%2C%20nonch%C3%A9%20allo%20scopo%20di%20indicare%20il%20livello%20di%20un%20insegnamento%20linguistico%20negli%20ambiti%20pi%C3%B9%20disparati.)
[2]|Livelli del CEFR|[URL](https://www.coe.int/en/web/common-european-framework-reference-languages/table-1-cefr-3.3-common-reference-levels-global-scale#:~:text=PROFICIENT%20USER,prepared%20to%20help.)
[3]|Text Classification|[URL](https://www.sciencedirect.com/topics/mathematics/text-classification)



#**Riferimenti ai dataset**
Nota|Nome
----|------
[1]|EFCAMDAT\|Shatz, I. (2020). Refining and modifying the EFCAMDAT: Lessons from creating a new corpus from an existing large-scale English learner language database. International Journal of Learner Corpus Research, 6(2), 220-236.




#**Altri notebook**
Numero|Titolo|URL
----|------|-------
**1 / 3**|**Introduzione, gestione dei dataset e preprocessing**|[link](https://colab.research.google.com/drive/11klGGwytqSNmE_WaWieAbVbtZL_qiWkl?authuser=4#scrollTo=75d3EdOJariY)
2 / 3|Costruzione dei modelli, training e primo testing|[link](https://colab.research.google.com/drive/1XXkGnLBFJmKjBbNl1a-1Q-73PIfT2QpZ?authuser=4#scrollTo=2PgXRu6P2iTy)
3 / 3|Grafici e conclusioni|[link](https://colab.research.google.com/drive/1-jy9wDUCTJ5TB1xcsnP9B_6wdTFvktw6?authuser=4#scrollTo=TT9PH3RlpgML)

